# 🔐 Smart Contract Vulnerability Detector
## Fine-tuning `microsoft/codebert-base` với LoRA — **Phiên bản đã sửa lỗi toàn bộ**

| Thành phần | Chi tiết |
|---|---|
| Mô hình nền | `microsoft/codebert-base` |
| Kỹ thuật | LoRA (r=16, α=32) + Sequence Classification |
| Dữ liệu | `smart_contract_corpus.json` (key: `Code` / `Label`) |
| Chia tập | Train 80% · Val 10% · Test 10% |
| Xử lý code dài | Sliding Window cả **Safe** và **Vulnerable** (stride 256) |
| Mất cân bằng | Class-weighted loss (Safe≈10k vs Vuln≈2k → ~5:1) |
| Đánh giá | Majority Voting (5 prompt) + F1-score |

> **Các lỗi đã sửa so với phiên bản cũ:**
> 1. ✅ Key `Input`/`Output` → `Code`/`Label`
> 2. ✅ Làm sạch field `Code` — tách code Solidity thuần khỏi text mô tả
> 3. ✅ Sliding window áp dụng cho **cả hai nhãn** (Vulnerable không bị cắt mất lỗ hổng)
> 4. ✅ Class-weighted loss giải quyết mất cân bằng 5:1
> 5. ✅ Prompt không bị nhân đôi với nội dung mô tả đã có trong field Code

## ⚙️ 0. Cài đặt thư viện

In [1]:
# %%capture
# Ép buộc cài đặt torchao phiên bản >= 0.16.0 để sửa lỗi ImportError của peft
!pip install -q -U transformers datasets peft accelerate scikit-learn "torchao>=0.16.0"

# Kiểm tra version sau khi cài
import importlib

print("=== KIỂM TRA PHIÊN BẢN THƯ VIỆN ===")
for pkg in ["transformers", "peft", "accelerate", "torchao"]:
    try:
        module = importlib.import_module(pkg)
        v = getattr(module, "__version__", "Không xác định")
        print(f"  {pkg}: {v}")
    except ImportError:
        print(f"  {pkg}: Chưa cài đặt!")

print("\n✅ Cài đặt hoàn tất!")
print("⚠️ NẾU BẠN VỪA CẬP NHẬT TORCHAO TỪ BẢN CŨ (0.10.0), HÃY RESTART KERNEL TRƯỚC KHI CHẠY TIẾP!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.8 MB/s eta 0:00:00
=== KIỂM TRA PHIÊN BẢN THƯ VIỆN ===
  transformers: 5.7.0


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


  peft: 0.19.1
  accelerate: 1.13.0
  torchao: 0.17.0

✅ Cài đặt hoàn tất!
⚠️ NẾU BẠN VỪA CẬP NHẬT TORCHAO TỪ BẢN CŨ (0.10.0), HÃY RESTART KERNEL TRƯỚC KHI CHẠY TIẾP!


## 📦 1. Import thư viện & Cấu hình

In [2]:
import json, re, random
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME  = "microsoft/codebert-base"
LABEL2ID    = {"Safe": 0, "Vulnerable": 1}
ID2LABEL    = {0: "Safe", 1: "Vulnerable"}
MAX_TOKENS  = 512    # Giới hạn cứng BERT-based

# Token budget: 512 - ~100 (prompt) - 3 (special tokens [CLS][SEP][SEP])
CODE_MAX_TOKENS = 409
STRIDE          = 256   # Stride cho sliding window (cả Safe và Vulnerable)

_dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"✅ Import OK | Device: {_dev}")

✅ Import OK | Device: Tesla T4


## 📂 2. Nạp & Làm sạch dữ liệu

### Vấn đề trong dữ liệu gốc
Field `Code` tồn tại ở **2 dạng**:
- **Dạng 1 (2766 mẫu):** Text mô tả + khối code Solidity trong backtick + phần "### As a Caller"
- **Dạng 2 (9612 mẫu):** Code Solidity thuần túy

Hàm `extract_solidity_code` xử lý cả hai dạng.

> **Lưu ý Kaggle:** Upload `smart_contract_corpus.json` vào Datasets rồi cập nhật `DATA_PATH`.


In [3]:
# ============================================================
# Cập nhật đường dẫn file JSON tại đây
# ============================================================
DATA_PATH = "/kaggle/input/datasets/thanhphuocjr/dapp-solo-dataset/smart_contract_corpus.json"

# -----------------------------------------------------------
# Hàm làm sạch: tách code Solidity thuần khỏi text mô tả
# -----------------------------------------------------------
def extract_solidity_code(text: str) -> str:
    """
    Xử lý 2 dạng dữ liệu:
    - Dạng 1: Có khối backtick  → trích xuất code giữa ``` ... ```
    - Dạng 2: Code thuần        → trả về nguyên bản (sau khi strip caller)

    Cũng loại bỏ phần '### As a Caller' nếu có.
    """
    # Dạng 1: Tìm khối code trong backtick (xử lý typo 'Solidiy')
    match = re.search(
        r'```(?:Solidit?y|solidity|Solidiy|sol)?\s*\n?(.*?)```',
        text,
        re.DOTALL | re.IGNORECASE,
    )
    if match:
        return match.group(1).strip()

    # Dạng 2: Code thuần — chỉ cắt phần '### As a Caller' nếu có
    text = re.split(r'###\s*As a Caller', text)[0]
    return text.strip()


# -----------------------------------------------------------
# Nạp dữ liệu
# -----------------------------------------------------------
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# FIX: Chuẩn hoá key → luôn dùng 'code' và 'label' nội bộ
cleaned_data = []
for item in raw_data:
    code_raw = item.get("Code") or item.get("Input", "")
    label    = item.get("Label") or item.get("Output", "")
    code_clean = extract_solidity_code(code_raw)
    if code_clean and label in LABEL2ID:
        cleaned_data.append({"code": code_clean, "label": label})

n_total = len(cleaned_data)
n_vuln  = sum(1 for d in cleaned_data if d["label"] == "Vulnerable")
n_safe  = n_total - n_vuln

print(f"Tổng mẫu sau làm sạch : {n_total:,}")
print(f"  Vulnerable           : {n_vuln:,}  ({n_vuln / n_total * 100:.1f}%)")
print(f"  Safe                 : {n_safe:,}  ({n_safe / n_total * 100:.1f}%)")
print(f"  Tỉ lệ mất cân bằng  : ~{n_safe / n_vuln:.1f}:1 (Safe:Vuln)")

print("\n--- Ví dụ mẫu 0 ---")
print(f"Label: {cleaned_data[0]['label']}")
print("Code (300 ký tự):", cleaned_data[0]['code'][:300])


Tổng mẫu sau làm sạch : 12,378
  Vulnerable           : 2,018  (16.3%)
  Safe                 : 10,360  (83.7%)
  Tỉ lệ mất cân bằng  : ~5.1:1 (Safe:Vuln)

--- Ví dụ mẫu 0 ---
Label: Safe
Code (300 ký tự): function vest( address to, uint256 amount, uint256 vestPeriodInSeconds ) external returns (uint256 vestIdx) { require(vestPeriodInSeconds > 0, "Vesting: vestPeriodInSeconds == 0"); token.safeTransferFrom(msg.sender, address(this), amount); vestIdx = accountVestList[to].length; accountVestList[to].pu


## 🎲 3. Kỹ thuật 5 Biến thể Prompt

In [4]:
def extract_fn_and_contract(code: str):
    """Trích xuất tên function và contract từ mã Solidity."""
    fn_match       = re.search(r'function\s+(\w+)\s*\(', code)
    contract_match = re.search(r'contract\s+(\w+)', code)
    fn_name       = fn_match.group(1)       if fn_match       else "unknownFunction"
    contract_name = contract_match.group(1) if contract_match else "UnknownContract"
    return fn_name, contract_name


PROMPT_VARIANTS = [
    {
        "a": "Devise a label name suitable for categorizing items as either vulnerable or safe.",
        "b": "Please review the code. Please find out if it is vulnerable.",
        "c": "The function {fn_name} from the contract {contract_name}.",
    },
    {
        "a": "Suggest a label designation that clearly identifies an item's status as either vulnerable or safe.",
        "b": "Inspect the following Solidity code. Determine if there are any vulnerabilities present.",
        "c": "Observe the method {fn_name} within the smart contract {contract_name}.",
    },
    {
        "a": "Invent a naming label that aptly segregates items into vulnerable or safe classifications.",
        "b": "Examine this Solidity script. Identify any potential security risks.",
        "c": "Review the function {fn_name} in the blockchain contract {contract_name}.",
    },
    {
        "a": "Formulate a label descriptor that bifurcates objects into categories of vulnerable and safe.",
        "b": "Please assess the provided Solidity code for any security vulnerabilities.",
        "c": "Check the procedure {fn_name} in the digital contract {contract_name}.",
    },
    {
        "a": "Propose a label nomenclature that aptly differentiates between vulnerable and safe states.",
        "b": "Evaluate the given Solidity function. Are there any security flaws?",
        "c": "Inspect the subroutine {fn_name} from the decentralized contract {contract_name}.",
    },
]


def build_prompt_text(code: str, variant_idx: int = None) -> str:
    """
    Tạo phần text_a (instruction) cho BERT text-pair input.
    variant_idx=None  → random 1 trong 5 (dùng khi train)
    variant_idx=0..4  → biến thể cố định (dùng khi val/test)
    """
    v = (PROMPT_VARIANTS[variant_idx]
         if variant_idx is not None
         else random.choice(PROMPT_VARIANTS))
    fn_name, contract_name = extract_fn_and_contract(code)
    c_filled = v["c"].format(fn_name=fn_name, contract_name=contract_name)
    return f"{v['a']} {v['b']} {c_filled}"


# Kiểm tra nhanh
for _i in [0, 2]:
    _t = build_prompt_text(cleaned_data[0]["code"][:150], variant_idx=_i)
    print(f"=== Biến thể {_i + 1} ===")
    print(_t)
    print()


=== Biến thể 1 ===
Devise a label name suitable for categorizing items as either vulnerable or safe. Please review the code. Please find out if it is vulnerable. The function vest from the contract UnknownContract.

=== Biến thể 3 ===
Invent a naming label that aptly segregates items into vulnerable or safe classifications. Examine this Solidity script. Identify any potential security risks. Review the function vest in the blockchain contract UnknownContract.



## 🔤 4. Nạp Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f"✅ Tokenizer loaded | vocab_size = {tokenizer.vocab_size:,}")
print(f"   model_max_length = {tokenizer.model_max_length}")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✅ Tokenizer loaded | vocab_size = 50,265
   model_max_length = 512


## 🪟 5. Sliding Window — áp dụng cho **CẢ HAI** nhãn

### Tại sao phải thay đổi so với phiên bản cũ?

Phiên bản cũ **truncate** Vulnerable tại 409 tokens. Nếu lỗ hổng nằm ở cuối hàm (ví dụ: reentrancy guard thiếu ở cuối), model sẽ đọc code an toàn nhưng bị ép học nhãn `Vulnerable` → **học ngược hoàn toàn**.

**Giải pháp:** Áp dụng sliding window cho **cả hai nhãn**:
- Mỗi chunk của mẫu Vulnerable đều mang nhãn `Vulnerable`
- Đảm bảo mọi phần của code đều được model thấy


In [6]:
def apply_sliding_window(
    tokenizer,
    code: str,
    label: str,
    max_tokens: int = CODE_MAX_TOKENS,
    stride: int = STRIDE,
) -> list:
    """
    Sliding window áp dụng cho CẢ HAI nhãn Safe và Vulnerable.

    - Nếu code ngắn hơn max_tokens → trả về 1 chunk nguyên bản
    - Nếu code dài hơn max_tokens  → chia thành các chunk chồng lấn
    - Tất cả chunk đều giữ nguyên nhãn gốc

    Args:
        tokenizer : HuggingFace tokenizer
        code      : mã nguồn Solidity đã làm sạch
        label     : "Vulnerable" hoặc "Safe"
        max_tokens: kích thước cửa sổ token tối đa cho code
        stride    : bước trượt (tokens)
    Returns:
        list[(chunk_code: str, label: str)]
    """
    token_ids = tokenizer.encode(code, add_special_tokens=False)

    # Code đủ ngắn → không cần slide
    if len(token_ids) <= max_tokens:
        return [(code, label)]

    chunks = []
    start  = 0
    while start < len(token_ids):
        end        = min(start + max_tokens, len(token_ids))
        chunk_code = tokenizer.decode(token_ids[start:end], skip_special_tokens=True)
        chunks.append((chunk_code, label))
        if end >= len(token_ids):
            break
        start += stride
    return chunks


# ── Kiểm tra ──
print("=== Kiểm tra Sliding Window ===")
for lbl in ["Safe", "Vulnerable"]:
    long_sample = max(
        (d for d in cleaned_data if d["label"] == lbl),
        key=lambda x: len(tokenizer.encode(x["code"], add_special_tokens=False))
    )
    tlen   = len(tokenizer.encode(long_sample["code"], add_special_tokens=False))
    chunks = apply_sliding_window(tokenizer, long_sample["code"], lbl)
    print(f"\n[{lbl}] Mẫu dài nhất: {tlen} tokens → {len(chunks)} chunk(s)")
    for i, (c, l) in enumerate(chunks):
        ct = len(tokenizer.encode(c, add_special_tokens=False))
        print(f"  Chunk {i}: {ct} tokens | label={l}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (573 > 512). Running this sequence through the model will result in indexing errors


=== Kiểm tra Sliding Window ===

[Safe] Mẫu dài nhất: 97823 tokens → 382 chunk(s)
  Chunk 0: 409 tokens | label=Safe
  Chunk 1: 409 tokens | label=Safe
  Chunk 2: 409 tokens | label=Safe
  Chunk 3: 409 tokens | label=Safe
  Chunk 4: 409 tokens | label=Safe
  Chunk 5: 409 tokens | label=Safe
  Chunk 6: 409 tokens | label=Safe
  Chunk 7: 409 tokens | label=Safe
  Chunk 8: 409 tokens | label=Safe
  Chunk 9: 409 tokens | label=Safe
  Chunk 10: 409 tokens | label=Safe
  Chunk 11: 409 tokens | label=Safe
  Chunk 12: 409 tokens | label=Safe
  Chunk 13: 409 tokens | label=Safe
  Chunk 14: 409 tokens | label=Safe
  Chunk 15: 409 tokens | label=Safe
  Chunk 16: 409 tokens | label=Safe
  Chunk 17: 409 tokens | label=Safe
  Chunk 18: 409 tokens | label=Safe
  Chunk 19: 409 tokens | label=Safe
  Chunk 20: 409 tokens | label=Safe
  Chunk 21: 409 tokens | label=Safe
  Chunk 22: 409 tokens | label=Safe
  Chunk 23: 409 tokens | label=Safe
  Chunk 24: 409 tokens | label=Safe
  Chunk 25: 409 tokens | lab

## ✂️ 6. Chia tập & Xây dựng HuggingFace Dataset

In [7]:
# Shuffle & split 80/10/10
random.shuffle(cleaned_data)
n       = len(cleaned_data)
n_train = int(0.8 * n)
n_val   = int(0.9 * n)

train_raw = cleaned_data[:n_train]
val_raw   = cleaned_data[n_train:n_val]
test_raw  = cleaned_data[n_val:]

print(f"Train raw : {len(train_raw):,}")
print(f"Val raw   : {len(val_raw):,}")
print(f"Test raw  : {len(test_raw):,}")


def tokenize_sample(code: str, prompt_text: str, label_str: str) -> dict:
    """
    Tokenize một mẫu theo dạng BERT text-pair:
        text_a = prompt_text (instruction)
        text_b = code        (Solidity source)
    Tokenizer tự thêm [CLS] ... [SEP] ... [SEP].
    """
    encoding = tokenizer(
        prompt_text,
        code,
        max_length=MAX_TOKENS,
        padding="max_length",
        truncation=True,
        return_tensors=None,
    )
    encoding["labels"] = LABEL2ID[label_str]
    return encoding


def build_hf_dataset(
    raw_items: list,
    use_sliding_window: bool = False,
    random_prompt: bool = True,
    fixed_variant_idx: int = 0,
) -> Dataset:
    """Chuyển raw_items → HuggingFace Dataset đã tokenize."""
    records = []
    for item in raw_items:
        code_src = item["code"]
        label    = item["label"]
        chunks   = (
            apply_sliding_window(tokenizer, code_src, label)
            if use_sliding_window else [(code_src, label)]
        )
        for chunk_code, chunk_label in chunks:
            v_idx       = None if random_prompt else fixed_variant_idx
            prompt_text = build_prompt_text(chunk_code, variant_idx=v_idx)
            enc         = tokenize_sample(chunk_code, prompt_text, chunk_label)
            records.append(enc)

    ds = Dataset.from_list(records)
    ds.set_format(type="torch")
    return ds


print("\nĐang build datasets (có thể mất vài phút)...")
train_dataset = build_hf_dataset(
    train_raw, use_sliding_window=True, random_prompt=True
)
val_dataset = build_hf_dataset(
    val_raw, use_sliding_window=True, random_prompt=False, fixed_variant_idx=0
)

print(f"\nTrain samples (sau sliding window): {len(train_dataset):,}")
print(f"Val   samples (sau sliding window): {len(val_dataset):,}")
print(f"Test  samples (raw)               : {len(test_raw):,}")
print("\n-- Ví dụ train[0] keys --")
print({k: v.shape for k, v in train_dataset[0].items()})

# Đếm nhãn sau sliding window để kiểm tra class weight
train_labels = [train_dataset[i]["labels"].item() for i in range(len(train_dataset))]
n_safe_train = train_labels.count(LABEL2ID["Safe"])
n_vuln_train = train_labels.count(LABEL2ID["Vulnerable"])
print(f"\nTrain label distribution:")
print(f"  Safe      : {n_safe_train:,}")
print(f"  Vulnerable: {n_vuln_train:,}")
print(f"  Tỉ lệ     : {n_safe_train / n_vuln_train:.2f}:1")


Train raw : 9,902
Val raw   : 1,238
Test raw  : 1,238

Đang build datasets (có thể mất vài phút)...

Train samples (sau sliding window): 52,289
Val   samples (sau sliding window): 6,085
Test  samples (raw)               : 1,238

-- Ví dụ train[0] keys --
{'input_ids': torch.Size([512]), 'attention_mask': torch.Size([512]), 'labels': torch.Size([])}

Train label distribution:
  Safe      : 50,372
  Vulnerable: 1,917
  Tỉ lệ     : 26.28:1


## 🤖 7. Khởi tạo Mô hình với LoRA + Class-Weighted Loss

### Class Weighting
Tỉ lệ Safe:Vulnerable ≈ 5:1 → nếu không xử lý, model sẽ thiên vị nhãn `Safe`.
Ta tính class weight nghịch đảo và truyền vào CrossEntropyLoss:

```
w_safe = n_vuln / n_total  →  nhỏ hơn
w_vuln = n_safe / n_total  →  lớn hơn
```


In [8]:
# ── Load base model ──
print(f"Đang tải {MODEL_NAME} ...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    trust_remote_code=True,
)

# ── LoRA config ──
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "key", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Đảm bảo classification head cũng được train
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

# ── Tính class weights ──
n_total_train = n_safe_train + n_vuln_train
# weight[class] = n_other / n_total  (cân bằng tương đối)
class_weights = torch.tensor(
    [n_vuln_train / n_total_train,   # weight cho Safe   (thấp hơn)
     n_safe_train / n_total_train],  # weight cho Vuln   (cao hơn)
    dtype=torch.float,
)
print(f"\nClass weights: Safe={class_weights[0]:.4f} | Vulnerable={class_weights[1]:.4f}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = class_weights.to(device)
model = model.to(device)
print(f"✅ Mô hình CodeBERT + LoRA sẵn sàng trên {device}!")


# ── Custom Trainer với class-weighted loss ──
class WeightedTrainer(Trainer):
    """
    Trainer tuỳ chỉnh để dùng class-weighted CrossEntropyLoss.
    Giải quyết mất cân bằng Safe:Vulnerable ≈ 5:1.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss     = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


Đang tải microsoft/codebert-base ...


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,476,866 || all params: 126,124,036 || trainable%: 1.1710

Class weights: Safe=0.0367 | Vulnerable=0.9633
✅ Mô hình CodeBERT + LoRA sẵn sàng trên cuda!


## 🏋️ 8. Huấn luyện

In [9]:
import os, glob

def compute_metrics(eval_pred):
    """Tính F1 class Vulnerable trong quá trình training."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_vuln = f1_score(labels, preds, pos_label=LABEL2ID["Vulnerable"], average="binary")
    f1_safe = f1_score(labels, preds, pos_label=LABEL2ID["Safe"], average="binary")
    return {"f1_vulnerable": f1_vuln, "f1_safe": f1_safe}


OUTPUT_DIR = "./codebert-vuln-lora"

def find_last_checkpoint(output_dir: str):
    ckpt_dirs = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not ckpt_dirs:
        return None
    ckpt_dirs = sorted(ckpt_dirs, key=lambda x: int(x.split("-")[-1]))
    return ckpt_dirs[-1]


last_checkpoint = find_last_checkpoint(OUTPUT_DIR)
if last_checkpoint:
    print(f"🔁 Tìm thấy checkpoint: {last_checkpoint} → Resume training")
else:
    print("🆕 Không có checkpoint → Train từ đầu")


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_vulnerable",
    greater_is_better=True,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="adamw_torch",
    dataloader_pin_memory=False,
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n🚀 Bắt đầu huấn luyện...")
trainer.train(resume_from_checkpoint=last_checkpoint)
print("\n✅ Huấn luyện hoàn tất!")


🆕 Không có checkpoint → Train từ đầu

🚀 Bắt đầu huấn luyện...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,F1 Vulnerable,F1 Safe
200,0.697562,0.227172,0.595318,0.979087
400,0.334216,0.257261,0.609677,0.979048
600,0.431123,0.150468,0.635634,0.980341
800,0.482841,0.188205,0.622735,0.980195
1000,0.301873,0.143384,0.617737,0.978291
1200,0.292371,0.143068,0.612368,0.977666



✅ Huấn luyện hoàn tất!


## 💾 9. Lưu Adapter LoRA

In [10]:
SAVE_PATH = "./codebert-vuln-lora-final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Adapter LoRA đã lưu tại: {SAVE_PATH}")


✅ Adapter LoRA đã lưu tại: ./codebert-vuln-lora-final


## 🗳️ 10. Đánh giá — Majority Voting trên tập Test

**Quy trình cho mỗi mẫu test:**
1. Chạy qua **5 biến thể prompt** của cùng đoạn code
2. Thu 5 dự đoán (logits → argmax)
3. Nếu ≥ 3 dự đoán là `Vulnerable` → nhãn cuối = `Vulnerable`
4. Tính F1 và Classification Report từ sklearn


In [11]:
model.eval()
model.to(device)

def predict_one(code: str, variant_idx: int) -> int:
    """Forward pass một mẫu với biến thể prompt cụ thể. Trả về label id."""
    prompt_text = build_prompt_text(code, variant_idx=variant_idx)
    enc = tokenizer(
        prompt_text,
        code,
        max_length=MAX_TOKENS,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    return int(torch.argmax(logits, dim=-1).item())


def majority_vote(code: str, n_variants: int = 5) -> int:
    """Chạy n_variants biến thể → majority vote → trả về label id."""
    votes = [predict_one(code, i) for i in range(n_variants)]
    return 1 if votes.count(1) >= (n_variants // 2 + 1) else 0


print("🗳️ Đang đánh giá tập Test với Majority Voting...")
print(f"   Số mẫu test: {len(test_raw):,}")

y_true, y_pred = [], []
for idx, item in enumerate(test_raw):
    code    = item["code"]
    true_lbl = LABEL2ID[item["label"]]
    pred_lbl = majority_vote(code, n_variants=5)
    y_true.append(true_lbl)
    y_pred.append(pred_lbl)
    if (idx + 1) % 100 == 0:
        print(f"  [{idx + 1}/{len(test_raw)}] done...")

print("\n" + "="*55)
print("📊 KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG")
print("="*55)
print(classification_report(
    y_true, y_pred,
    target_names=["Safe", "Vulnerable"],
    digits=4,
))

f1_overall = f1_score(y_true, y_pred, average="macro")
f1_vuln    = f1_score(y_true, y_pred, pos_label=1, average="binary")
print(f"F1 Macro   : {f1_overall:.4f}")
print(f"F1 Vuln    : {f1_vuln:.4f}")


🗳️ Đang đánh giá tập Test với Majority Voting...
   Số mẫu test: 1,238
  [100/1238] done...
  [200/1238] done...
  [300/1238] done...
  [400/1238] done...
  [500/1238] done...
  [600/1238] done...
  [700/1238] done...
  [800/1238] done...
  [900/1238] done...
  [1000/1238] done...
  [1100/1238] done...
  [1200/1238] done...

📊 KẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG
              precision    recall  f1-score   support

        Safe     0.9977    0.8346    0.9089      1040
  Vulnerable     0.5326    0.9899    0.6926       198

    accuracy                         0.8595      1238
   macro avg     0.7652    0.9123    0.8007      1238
weighted avg     0.9233    0.8595    0.8743      1238

F1 Macro   : 0.8007
F1 Vuln    : 0.6926


## 🔍 11. Inference nhanh — Kiểm tra một hàm bất kỳ

In [12]:
TEST_CODE = '''
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool success,) = msg.sender.call{value: amount}("");
    require(success);
    balances[msg.sender] -= amount;
}
'''

pred_id   = majority_vote(TEST_CODE, n_variants=5)
pred_name = ID2LABEL[pred_id]
print(f"Dự đoán: {pred_name}")
print("(Hàm trên có lỗi reentrancy — balance giảm SAU khi transfer)")


Dự đoán: Vulnerable
(Hàm trên có lỗi reentrancy — balance giảm SAU khi transfer)
